# Notebook 01 — Model Training, Validation, and Prediction

## Purpose

This notebook reproduces the complete machine-learning pipeline used in this study. It prepares the curated literature dataset, computes or loads cached feature representations, trains the final gradient boosting regression (GBR) model, performs internal validation, evaluates the model on independent out-of-distribution (OOD) wet-lab ASOs, and exports all prediction and feature artifacts required by the downstream notebooks.

---

## Workflow

| Cell | Description |
|------|-------------|
| 1 | Repository setup, configuration, and directory initialization |
| 2 | Runtime configuration |
| 3 | Input validation and preflight checks |
| 4 | Import project modules |
| 5 | Output directory initialization |
| 6 | Define the complete training pipeline and helper functions |
| 7 | Define the final GBR model wrapper |
| 8 | Load or regenerate cached feature matrices |
| 9 | Train the final GBR model |
| 10 | Internal cross-validation and model performance assessment |
| 11 | Independent out-of-distribution prediction on wet-lab ASOs |
| 12 | Export predictions, metrics, feature artifacts, and Figure 4 progression values for downstream notebooks |

---

## Outputs

This notebook writes the following reproducible outputs:

- Trained model artifacts
- Internal validation metrics
- Out-of-distribution predictions
- Cached feature matrices
- Figure 4 feature-ablation summary


In [1]:
# ============================================================
# PROJECT SETUP
# ============================================================

import os
import sys
from pathlib import Path

# ------------------------------------------------------------
# Locate repository root
# Notebook is expected to live in:
#
# repo-root/
# ├── data/
# ├── features/
# ├── notebooks/
# ├── outputs/
# ├── src/
# └── config.yaml
# ------------------------------------------------------------

CWD = Path.cwd().resolve()

candidates = [
    CWD,          # if notebook is launched from repo root
    CWD.parent,   # if notebook is launched from notebooks/
]

ROOT = None

for candidate in candidates:
    if (
        (candidate / "src").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "features").is_dir()
        and (candidate / "config.yaml").is_file()
    ):
        ROOT = candidate.resolve()
        break

if ROOT is None:
    raise RuntimeError(
        "Could not locate repository root.\n"
        "Expected folders: src/, data/, features/ and config.yaml\n"
        f"Notebook launched from: {CWD}"
    )

# Make all relative paths in legacy code resolve from repo root
os.chdir(ROOT)

# Make src modules importable
SRC_DIR = ROOT / "src"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------

from config import load_config

cfg = load_config(
    ROOT / "config.yaml",
    project_root_override=ROOT
)

# Canonical project directories
DATA_DIR = ROOT / "data"
FEAT_DIR = ROOT / "features"

OUTPUT_DIR = ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "model"
FIG_DIR = OUTPUT_DIR / "figures"
THERMO_CACHE_DIR = OUTPUT_DIR / "thermo_cache"
RESULTS_DIR_BASE = OUTPUT_DIR / "results"
SUPP_TABLE_DIR = OUTPUT_DIR / "supplementary_tables"

for d in [
    OUTPUT_DIR,
    MODEL_DIR,
    FIG_DIR,
    THERMO_CACHE_DIR,
    RESULTS_DIR_BASE,
    SUPP_TABLE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = ROOT
ROOT_DIR = ROOT

print("Repository root :", ROOT)
print("Data directory  :", DATA_DIR)
print("Feature directory:", FEAT_DIR)
print("Output directory:", OUTPUT_DIR)

Repository root : /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy
Data directory  : /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/data
Feature directory: /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/features
Output directory: /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/outputs


In [2]:
# === RUN CONFIG (column overrides etc.) ===
THERMO_ASO_COL_OVERRIDE    = "Sequence"
THERMO_TARGET_COL_OVERRIDE = "target_window"  # not "target"
print("[thermo][override] ASO =", THERMO_ASO_COL_OVERRIDE,
      " TARGET =", THERMO_TARGET_COL_OVERRIDE)


[thermo][override] ASO = Sequence  TARGET = target_window


In [3]:
# === Preflight

from preflight import check_inputs, print_lab_columns
from pathlib import Path

need = ["relative_expression.csv", "DAZ_top5.tsv", "FAM_top5.tsv"]
exists = check_inputs(need, base=cfg.paths.data)
print({k: bool(v) for k,v in exists.items()})
print_lab_columns(cfg.paths.data / "relative_expression.csv")


{'relative_expression.csv': True, 'DAZ_top5.tsv': True, 'FAM_top5.tsv': True}
[lab columns]: ['target', 'ID', 'group', 'rel_expr']


['target', 'ID', 'group', 'rel_expr']

In [4]:
# == Import modules

import sys, pathlib
from pathlib import Path
import json, pandas as pd, numpy as np

# Modules for downstream steps
from utils import make_run_id, run_paths, compute_map_sig, build_results_dir, safe_mkdir, assert_safe_to_write, config_get
from io_frames import build_merged
from features import ensure_sequence_column, finalize_and_save_features
from design_matrix import choose_and_build_design
from synthetic import make_synthetic_training
from thermo import compute_thermo_features, thermo_upgrade_section3
from mapping import map_and_access  
from train_eval import (
    fit_ridge_logit, loocv_metrics_real_only, relative_loocv_logo,
    run_ablation_and_logo, save_calibration_plot, loocv_mae_r2, 
    rmse, calibration_stats, permutation_baseline_spearman, summarize_logo_csv
)
from models import add_partial_pooling_X
from train_eval_resid import fit_ridge_residualized_cv, fit_pairwise_logit_cv

print("Model modules loaded successfully.")
#----------------------------------------------------------

Model modules loaded successfully.


In [5]:
#---------------- Directory setup for Phase 8+  ---------------------
results_root = RESULTS_DIR_BASE
phase = cfg.extras.get("PHASE", "P8")
exp_tag = cfg.extras.get("EXPERIMENT_TAG","p8a_gene_resid_core")
timestamped = bool(cfg.extras.get("TIMESTAMPED_SUBDIRS", True))
refuse_overwrite = bool(cfg.extras.get("REFUSE_OVERWRITE", False))

RESULTS_DIR = build_results_dir(results_root, phase, exp_tag, timestamped)

features_out = RESULTS_DIR / f"{phase.lower()}_{exp_tag}_features.csv"
print("Repository root:", ROOT_DIR)
print("Results directory:", RESULTS_DIR)

#----------------------------------------------------------

Repository root: /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy
Results directory: /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/outputs/results/P8/p8a_gene_resid_core/20260812_113411


In [6]:

# ========= MASTER RUNNER (explicit Steps 1 → 8) =========
import pathlib
from pathlib import Path

def run_pipeline(
    sim_mode: str = "basic",                # "basic" | "enhanced" | "synthetic"
    relative: bool = False,                 # absolute KD vs relative-KD evaluation
    ridge_alpha: float = 1.0,
    synth_n_per_real: int = 100,
    synth_weight: float = 0.3,
    noise_sigma0: float = 0.03, noise_c1: float = 0.07, noise_c2: float = 0.05,
    run_ablation: bool = True,
    run_logo: bool = True,
    full_bio: bool = False,                 # enforce Vienna-only in thermo
    thermo_cache: bool = True,              # use jsonl cache for thermo
    force_recompute_thermo: bool = False,
    verbose: bool = True,
    ROOT: Path | None = None,
    phase: str | None = None,               # e.g., "P1", "P2", etc. (for run_id; optional)
    basic_feature_set: str | None = None,   # "proxy_only"|"proxy_duplex"|"proxy_duplex_access" (Phase-1 convenience)
    feat_path_override: Path | None = None, # flag for using basic snapshot for Phase1 to 3
    gold_path_override=None,                 # flag for using thermo snapshot - pass GOLD_FEATS here for P4–P6
    label_transform: str | None = None,
    calibration: str = "linear",
    calibration_scope: str = "global",      # for Phase 9
    partial_pooling: str | None = None,
):
    
    # Use repository root
    if ROOT is None:
        ROOT = PROJECT_ROOT
    else:
        ROOT = Path(ROOT).resolve()
    
    import json, pandas as pd, numpy as np
    
    sim_mode = sim_mode.lower().strip()
    assert sim_mode in {"basic","enhanced","synthetic"}, "sim_mode must be basic/enhanced/synthetic"
    if verbose:
        print(f"\n=== RUN PIPELINE ===\nmode={sim_mode}, relative={relative}\n")

    DATA_DIR = ROOT / "data"
    FEAT_DIR = ROOT / "features"

    OUTPUT_DIR = ROOT / "outputs"
    MODEL_DIR = OUTPUT_DIR / "model"
    FIG_DIR = OUTPUT_DIR / "figures"
    THERMO_CACHE_DIR = OUTPUT_DIR / "thermo_cache"

    for d in [MODEL_DIR, FIG_DIR, THERMO_CACHE_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    # normalize calibration early
    calibration = (calibration or "linear").lower()
    if calibration not in ("linear", "isotonic"):
        print(f"[warn] unknown calibration={calibration}; defaulting to 'linear'")
        calibration = "linear"
    print("[debug] calibration:", calibration, "feat_path_override:", feat_path_override)
    
    # normalize calibration scope early
    calibration_scope = (calibration_scope or "global").lower()
    if calibration_scope not in {"global", "by_family", "by_gene"}:
        print(f"[warn] unknown calibration_scope={calibration_scope}; defaulting to 'global'")
        calibration_scope = "global"
    print("[debug] calibration_scope:", calibration_scope)
    
    # Alias arg name used by drivers
    feature_set = basic_feature_set

    # ---------------- STEP 1 — Merge lab + designs ----------------
    # Reads LAB_CSV and the design TSVs from Master_model/data, writes merged_aso_results.csv
    merged = build_merged(
        lab_csv=DATA_DIR / "relative_expression.csv",
        designs=[DATA_DIR / "DAZ_top5.tsv", DATA_DIR / "FAM_top5.tsv"],
        data_dir=DATA_DIR,
        verbose=verbose,
    )

    print(f"[run_pipeline] label_transform={label_transform}")

    # ---------------- Prepare transcript FASTA paths ----------------
    # Your earlier lists were bare filenames; prefix them with DATA_DIR so mapping can find them.
    TRANSCRIPTS = {
        "DAZ": [
            DATA_DIR / "DAZ_MANE.fasta",
            DATA_DIR / "DAZ_A.fasta",
            DATA_DIR / "DAZ_B.fasta",
            DATA_DIR / "DAZ_C.fasta",
            DATA_DIR / "DAZ_D.fasta",
            DATA_DIR / "DAZ_E.fasta",
            DATA_DIR / "DAZ_F.fasta",
            DATA_DIR / "DAZ_G.fasta",
        ],

        "FAM": [
            DATA_DIR / "FAM_MANE.fasta",
            DATA_DIR / "FAM_A.fasta",
            DATA_DIR / "FAM_B.fasta",
            DATA_DIR / "FAM_C.fasta",
            DATA_DIR / "FAM_D.fasta",
        ],
        "TARDBP": [
            DATA_DIR/"TARDBP_NM_007375_MANE_Select.fasta",
        ],
        "SOD1": [
        DATA_DIR/"SOD1_MANE.fasta",
        ],
        "ATXN2": [
            DATA_DIR/"ATXN2_MANE.fasta",
        ],
        "C9ORF72": [
            DATA_DIR/"C9ORF72_MANE.fasta",
        ],
        "FUS": [
            DATA_DIR/"FUS_MANE.fasta",
        ],
        "STMN2": [
            DATA_DIR/"STMN2_MANE.fasta",
        ],
        "AR": [
            DATA_DIR/"AR_MANE.fasta",
        ],
        "UNC13A": [
            DATA_DIR/"UNC13A_MANE.fasta",
        ],
    }
    ALIASES = {
        "DAZ": "DAZ",
        "FAM": "FAM",
        "TARDBP": "TARDBP", "TDP-43": "TARDBP", "TDP43": "TARDBP",
    }
    # Training is inlined inside run_pipeline, just skip any Phase-9 / Step-3 code paths once 
    # feat_path_override is set, and drop directly into the “train” block.
    if feat_path_override:
        feat_path = Path(feat_path_override)
        df = pd.read_csv(feat_path)  # keep df in scope for later
        print(f"[Step-2] Using precomputed features from {feat_path}")

        # Fast-path flags: skip mapping/plfold and skip Step-3 thermo upgrade
        skip_step2_mapping = True
        skip_step3_thermo = True
    else:
        skip_step2_mapping = False
        skip_step3_thermo = False

    # ---------------- STEP 0.9 — Prepare run directories EARLY (needed by later steps) ----------------
    RUNS_DIR  = MODEL_DIR / "runs"
    RUNS_DIR.mkdir(parents=True, exist_ok=True)

    # Build a run_id early (if you already compute run_id elsewhere, it's fine to reuse that; this is a safe fallback)
    from datetime import datetime
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    phase_tag   = str(phase) if phase is not None else "P"
    feat_tag    = (feature_set or "feat")
    alpha_tag   = f"a{ridge_alpha}".replace(".", "p") if "ridge_alpha" in locals() else "a"
    run_id      = f"{phase_tag}__{feat_tag}__{alpha_tag}__{stamp}"

    # Define run dir and ensure it exists
    MODEL_RUN_DIR = RUNS_DIR / run_id
    MODEL_RUN_DIR.mkdir(parents=True, exist_ok=True)

    # (Optional) helpful debug so you can see it once per run
    print("[debug] MODEL_RUN_DIR:", MODEL_RUN_DIR)

    # ---------------- STEP 2 — Mapping + accessibility + finalize features ----------------
    snap2 = None  # <-- initialize so it's always defined
    if feat_path_override:
        # ===== FAST PATH: use precomputed snapshot as-is =====
        feat_path = Path(feat_path_override)
        df = pd.read_csv(feat_path)  # keep df around for downstream
        print(f"[Step-2] FAST PATH: using precomputed features from {feat_path}")

        # IMPORTANT:
        # - Do NOT run mapping/plfold
        # - Do NOT call add_phase9_features on top of a snapshot
        # - Do NOT call finalize_and_save_features on a snapshot
        #
        # If you want to allow Phase-9 on top of a snapshot in the future,
        # do it in Step-4 (design/build) behind an explicit flag, not here.

        # Let Step-3 know we already have features; it should usually be skipped
        # (Guard Step-3 with `and not skip_step3_thermo` where you enter Step-3.)
        skip_step3_thermo = True

    else:
        # ===== SLOW PATH: compute mapping/plfold, then finalize features =====
        df = ensure_sequence_column(merged)

        df = map_and_access(
            df,
            transcripts=TRANSCRIPTS,
            aliases=ALIASES,
            W=120, U=20,
            max_mismatches=3,
            use_duplex_fallback=True,
            compute_accessibility=True,
            verbose=verbose,
        )

        must = ["dG_open_mean", "dG_open_min", "neg_dG_eff", "target_gc_pm10", "target_gc_pm30"]
        print("missing:", [c for c in must if c not in df.columns])

        # Optional Phase-9 augmentation ONLY on freshly computed features
        if (feature_set or "").lower() == "p9_context":
            from features import add_phase9_features
            if not set(must).issubset(df.columns):
                df = add_phase9_features(
                    df,
                    context_windows=(10, 30),
                    k_orders=(2, 3),
                    kmer_pca_dims=10,
                )
                print("[P9] added features; now present:", [c for c in must if c in df.columns])
            else:
                print("[P9] features already present – skipping add_phase9_features()")

        # 2.4–2.5 finalize required columns & write features/aso_features.csv
        feat_path = finalize_and_save_features(
            df,
            feat_dir=FEAT_DIR,
            merged_for_kd=merged,
            verbose=verbose,
        )

        # ---- Step-2 snapshot (mapping/access provenance) ----
        MAP_SIG = compute_map_sig(df)
        max_mismatches_used = 3
        use_duplex_flag = 1
        use_access_flag = 1
        snap2 = FEAT_DIR / f"snapshots/aso_features_step2_map-{MAP_SIG}_mm{max_mismatches_used}_dup{use_duplex_flag}_acc{use_access_flag}.csv"
        snap2.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(snap2, index=False)

        # We computed features freshly, so Step-3 may run (thermo upgrade) if needed.
        skip_step3_thermo = False
        
        # 2.4–2.5 Fill required columns, merge KD if missing, and write features/aso_features.csv
        feat_path = finalize_and_save_features(
            df,
            feat_dir=FEAT_DIR,
            merged_for_kd=merged,
            verbose=verbose,
        )

        # ---- Step-2 snapshot (mapping/access) ----
        MAP_SIG = compute_map_sig(df)
        max_mismatches_used = 3
        use_duplex_flag = 1
        use_access_flag = 1
        snap2 = FEAT_DIR / f"snapshots/aso_features_step2_map-{MAP_SIG}_mm{max_mismatches_used}_dup{use_duplex_flag}_acc{use_access_flag}.csv"
        snap2.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(snap2, index=False)

    # ---------------- STEP 3 — Thermo upgrade (enhanced/synthetic only) ----------------
    # This faithfully reproduces your Section-3 (cache, Vienna-only guardrails, gold snapshot).
    gold_path = None
    print(f"[Step-3] gold_path_override set to: {gold_path_override}")
    
    def _has_thermo_columns(frame):
        cols = set(map(str.lower, frame.columns))
        return ("dg37_kcalmol" in cols) or ("neg_dg_bind" in cols and "duplex_dg" in cols)
    
    if sim_mode in {"enhanced", "synthetic"} and not skip_step3_thermo:
        # EARLY EXIT: if caller provided GOLD (or GOLD+kmers), do NOT thermo-upgrade or renormalize
        if gold_path_override is not None:
            feat_path = Path(gold_path_override)  # this is our active features file
            df = pd.read_csv(feat_path)
            # (optionally: light column sanity checks here)
            # then jump straight to Step-4
        else:
            # (A) If caller provided a gold snapshot, use it and skip compute.
            if gold_path_override is not None:
                gold_path = Path(gold_path_override)
                if not gold_path.exists():
                    raise FileNotFoundError(f"[Step-3] gold_path_override not found: {gold_path}")
                df = pd.read_csv(gold_path)
                feat_path = gold_path  # downstream uses this as the active feature set
                if verbose:
                    print(f"[Step-3] Using precomputed THERMO GOLD features from {gold_path}")

            # (B) Else, if current df already contains thermo columns, accept as gold.
            elif _has_thermo_columns(df):
                gold_path = feat_path  # whatever we loaded already has thermo
                if verbose:
                    print(f"[Step-3] Detected thermo columns in current features → treating as GOLD: {gold_path}")

            # (C) Else, upgrade Step-2 → thermo using helper (cache-aware)
            else:
                if verbose:
                    print("[Step-3] Upgrading Step-2 features to THERMO (ΔG°37) via thermo_upgrade_section3...")
                merged, thermo_feat_path, gold_path = thermo_upgrade_section3(
                    sim_mode=sim_mode,
                    feat_path=feat_path,
                    merged=merged,
                    features_dir=FEAT_DIR,
                    thermo_cache_dir=THERMO_CACHE_DIR,
                    full_bio=full_bio,
                    thermo_cache=thermo_cache,
                    force_recompute_thermo=force_recompute_thermo,
                    verbose=verbose,
                )
                # To train against gold_path directly, set:
                feat_path = Path(gold_path) if gold_path is not None else Path(feat_path)   # <<< ensure Path; DO NOT override with thermo_feat_path

        # ---- P10b: ensure required thermo-derived columns exist on-disk and in-memory ----
        feat_path = Path(feat_path)                                                     # <<< ensure Path for any branch (A/B/C)
        df = pd.read_csv(feat_path)

        # 1) neg_dG_bind = - dG37_kcalmol
        if "neg_dG_bind" not in df.columns:
            # Handle common naming variants just in case:
            dG_candidates = [c for c in df.columns if c.lower() in {"dg37_kcalmol", "dg37", "dg_duplex_37", "duplex_dg37"}]
            if not dG_candidates:
                raise KeyError("Thermo CSV missing ΔG°37 column; expected one of: dG37_kcalmol / dG37 / dg_duplex_37 / duplex_dG37")
            dG_col = dG_candidates[0]
            df["neg_dG_bind"] = -df[dG_col]

        # 2) (optional) neg_dG_eff = -(dG37 - dG_open_mean)
        if "neg_dG_eff" not in df.columns and "dG_open_mean" in df.columns:
            dG_col = [c for c in df.columns if c.lower() in {"dg37_kcalmol", "dg37", "dg_duplex_37", "duplex_dg37"}][0]
            df["neg_dG_eff"] = -(df[dG_col] - df["dG_open_mean"])

        # Normalize thermo column name (idempotent)
        if "dg37_kcalmol" in df.columns and "dG37_kcalmol" not in df.columns:
            df = df.rename(columns={"dg37_kcalmol": "dG37_kcalmol"})

        # Enforce neg_dG_bind = -dG37_kcalmol when dG37 is present
        if "dG37_kcalmol" in df.columns:
            df["dG37_kcalmol"] = pd.to_numeric(df["dG37_kcalmol"], errors="coerce")
            df["neg_dG_bind"]  = -df["dG37_kcalmol"]
            _diff = (df["neg_dG_bind"] + df["dG37_kcalmol"]).abs().max(skipna=True)
            if _diff > 1e-3 and verbose:
                print(f"[Step-3][note] normalized neg_dG_bind from dG37 (post-fix diff={_diff:.4f})")

            # --- write a normalized copy next to the original ---
            feat_path = Path(feat_path)
            if isinstance(NORMALIZED_SNAPSHOT_SUFFIX, str):
                out_path = feat_path.with_name(feat_path.stem + NORMALIZED_SNAPSHOT_SUFFIX + feat_path.suffix)
            else:
                out_path = feat_path.with_name(feat_path.stem + "__norm_v1" + feat_path.suffix)
            df.to_csv(out_path, index=False)
            if verbose:
                print(f"[Step-3] wrote normalized GOLD snapshot → {out_path}")
            feat_path = out_path                                        # <<< downstream uses the normalized file
        else:
            # Persist edits back to the same file if we didn't re-alias dG37
            df.to_csv(feat_path, index=False)

        # Re-read the active feature set path so downstream is consistent
        df = pd.read_csv(feat_path)

                    
    # ---------------- STEP 4 — Design matrix & scalers ----------------
    # If your thermo snapshot only has dG37, standardize neg_dG_bind now (safe no-op if present)
    if "dG37_kcalmol" in df.columns and "neg_dG_bind" not in df.columns:
        df["neg_dG_bind"] = -df["dG37_kcalmol"]

    # For synthetic runs, force relative=True to suppress target_* dummies
    relative_eff = True if (sim_mode == "synthetic") else relative

    # Build the design using your existing helper (this may add/transform columns)
    # If we had a snapshot override, force it here.
    if feat_path_override:
        feat_path = Path(feat_path_override)
    else:
        feat_path = Path(feat_path) if 'feat_path' in locals() and feat_path is not None else None

    if feat_path is not None:
        print(f"[run] Using snapshot features at {feat_path}")
        
        # >>> PHASE-9: augment old GOLD with P9 features, then point feat_path at the augmented CSV
        if (feature_set or "").lower() == "p9_context":
            from features import add_phase9_features
            need = {"dG_open_mean","dG_open_min","neg_dG_eff","aso_gc_frac","target_gc_pm10","target_gc_pm30"}

            # if the snapshot doesn't have P9 columns yet, add them
            if not need.issubset(set(df.columns)):
                df = add_phase9_features(df, context_windows=(10,30), k_orders=(2,3), kmer_pca_dims=10)
                # write the augmented frame to a run-local CSV and pass that into the design builder
                p9_feat_csv = Path(feat_path).with_name(Path(feat_path).stem + "__p9.csv")
                df.to_csv(p9_feat_csv, index=False)
                feat_path = p9_feat_csv
                print(f"[P9] augmented features written → {p9_feat_csv}")
    
    F, X_design, y, feature_order, core_feats = choose_and_build_design(
        feat_path=feat_path,
        model_dir=MODEL_RUN_DIR,
        relative=relative_eff,
        verbose=verbose,
        feature_set=feature_set,     # <— Phase 9
        use_gene_z=True,             # <— optional; on for P9
    )
    # >>> [OPTIONAL DEBUG]
    if (feature_set or "").lower() == "p9_context":
        print("[debug][train_eval] first cols:", X_design.columns.tolist()[:12], "… total:", len(X_design.columns))
    
    # --- Save a raw copy of F for relative evaluation later (Step-7a) ---
    F_raw_for_relative = F.copy()
    
    # >>> Add this block right here (before any calibration) <<<
    if "family" not in F_raw_for_relative.columns:
        # pick a source field we already have
        if "target" in F_raw_for_relative.columns:
            src = F_raw_for_relative["target"].astype(str)
        elif "gene" in F_raw_for_relative.columns:
            src = F_raw_for_relative["gene"].astype(str)
        else:
            src = None

        if src is None:
            F_raw_for_relative["family"] = "UNK"
        else:
            fam = src.str.upper().replace({
                "DAZ": "DAZ",
                "FAM": "FAM",
            })
            F_raw_for_relative["family"] = fam.fillna("UNK")

    # ---- Tier-based feature whitelist (do this ONCE, on X_design) ----    
    is_p9 = (str(basic_feature_set or "").lower() == "p9_context")
    is_proxy_mode = str(basic_feature_set or "").lower().startswith("proxy")
    if not is_p9 and is_proxy_mode:    
        def _feature_cols_for_set(basic_feature_set: str):
            cols = ["neg_dG_bind"]  # proxy in P1, thermo in P4+ (same name)
            fs = (basic_feature_set or "").lower()
            if "duplex" in fs:
                cols.append("neg_duplex_dG")
            if "access" in fs or "punp" in fs:
                cols.append("logit_punp")
            return cols

        strict = (sim_mode != "synthetic")  # be tolerant only in P7
        feature_cols = _feature_cols_for_set(basic_feature_set)

        # Force the thermo-era trio in enhanced/synthetic when feature_set is unspecified
        if sim_mode in {"enhanced", "synthetic"} and (basic_feature_set is None or str(basic_feature_set).strip()==""):
            feature_cols = ["neg_dG_bind", "neg_duplex_dG", "logit_punp"]

        missing = [c for c in feature_cols if c not in X_design.columns]
        if missing and strict:
            raise KeyError(f"Missing required features for set={basic_feature_set}: {missing}")
        elif missing:  # P7 synthetic: allow zero-fill
            for c in missing:
                if verbose: print(f"[design][synthetic] injecting missing column {c}=0.0")
                X_design[c] = 0.0

        # AFTER choose_and_build_design(...) and your whitelist on X_design
        feature_order = list(X_design.columns)  # overwrite to the whitelisted order
        core_feats    = list(X_design.columns)
        F = X_design.copy()  # make sure the "full frame" the trainer sees IS the whitelist

        # Drop any non-whitelisted columns so fit/predict schemas match exactly
        keep = [c for c in feature_cols if c in X_design.columns]
        X_design = X_design[keep]  # enforce order & subset

    # keep your Phase-1 convenience filters INSIDE the same if-not-is_p9 guard
    if not is_p9:
        # The whitelist already enforced exactly these subsets:
        # - proxy_only -> ["neg_dG_bind"]
        # - proxy_duplex -> ["neg_dG_bind","neg_duplex_dG"]
        # - proxy_duplex_access -> ["neg_dG_bind","neg_duplex_dG","logit_punp"]
        # ---- Phase-1 convenience filters  ----
        if basic_feature_set == "proxy_only":
            keep = ["neg_dG_bind"]; keep = [c for c in keep if c in X_design.columns]
            X_design = X_design[keep]; core_feats = keep[:]
        elif basic_feature_set == "proxy_duplex":
            keep = [c for c in ["neg_dG_bind","neg_duplex_dG"] if c in X_design.columns]
            X_design = X_design[keep]; core_feats = keep[:]
        elif basic_feature_set == "proxy_duplex_access":
            keep = [c for c in ["neg_dG_bind","neg_duplex_dG","logit_punp"] if c in X_design.columns]
            X_design = X_design[keep]; core_feats = keep[:]
    
    # --- Reset downstream variables after subsetting ---
    feature_order = list(X_design.columns)
    core_feats    = list(X_design.columns)
    F = X_design.copy()

    #print("[debug] sim_mode:", sim_mode,
    #    "feature_set:", basic_feature_set,
    #    "relative_eff:", relative_eff)
    #print("[debug] FINAL X_design cols:", list(X_design.columns))
    
    # Run ID + per-run folders
    use_duplex  = int("neg_duplex_dG" in X_design.columns)
    use_access  = int("logit_punp"    in X_design.columns)
    synthetic_tag = ("none" if sim_mode != "synthetic" else "A")
    run_id_base = make_run_id(
        phase=phase, mode=sim_mode, relative=relative_eff, alpha=ridge_alpha, ablation=bool(run_ablation),
        use_duplex=use_duplex, use_access=use_access, synthetic=synthetic_tag,
        synth_n_per_real=(synth_n_per_real if sim_mode=="synthetic" else None),
        synth_weight=(synth_weight if sim_mode=="synthetic" else None)
    )
    
    # --- finalize run_id (only add suffixes for Phase 8 variants) ---
    is_p8 = str(phase).lower().startswith("p8")

    if is_p8:
        if label_transform in ("pairwise_gene", "rank_gene", "pairwise"):      # P8b
            run_id = f"{run_id_base}__cal_{calibration}"                       # e.g., __cal_linear / __cal_isotonic
        elif label_transform in ("gene_resid", "residualize_gene", "resid_gene"):  # P8a residualized
            run_id = f"{run_id_base}__lt_resid"
        else:                                                                  # P8a identity (no transform)
            run_id = f"{run_id_base}__lt_identity"
    else:
        # All other phases (P1–P7, P9…) keep the original naming
        run_id = run_id_base

    # Create dirs ONCE, right after finalizing run_id
    MODEL_RUN_DIR, FIG_RUN_DIR = run_paths(OUTPUT_DIR, run_id)
    #print("[debug] final run_id:", run_id, "| MODEL_RUN_DIR:", MODEL_RUN_DIR.name)

    # (Optional check)
    if is_p8:
        if label_transform in ("pairwise_gene","rank_gene","pairwise"):
            assert "__cal_" in MODEL_RUN_DIR.name
        elif label_transform in ("gene_resid","residualize_gene","resid_gene"):
            assert "__lt_resid" in MODEL_RUN_DIR.name
        else:
            assert "__lt_identity" in MODEL_RUN_DIR.name
    
    # ---------------- PHASE 8a — Residualized labels + Ridge (conditional) ----------------
    if label_transform in ("gene_resid", "residualize_gene", "resid_gene"):
        # 8a.1 — Extract identifiers safely (gene, family) from the pre-whitelist frame
        def _safe_series(frame: pd.DataFrame, candidates, default):
            for c in candidates:
                if c in frame.columns:
                    return frame[c].astype(str).reset_index(drop=True)
            return pd.Series([default] * len(frame), dtype=str)

        gene_ids   = _safe_series(F_raw_for_relative, ["gene", "target_gene", "target"], default="UNK_GENE")
        family_ids = _safe_series(F_raw_for_relative, ["family", "target_family"], default="UNK_FAM")
        y_kd       = pd.Series(y).astype(float).reset_index(drop=True)

        # 8a.2 — Build LOGO folds by gene (or flip to LOOCV if you prefer)
        def make_logo_folds_by_gene(genes: pd.Series, min_train=2, min_test=1):
            folds = []
            n = len(genes)
            groups = genes.groupby(genes).groups
            for gname, idx in groups.items():
                test_idx  = np.array(sorted(list(idx)))
                train_idx = np.array(sorted(list(set(range(n)) - set(test_idx))))
                if len(test_idx) < min_test or len(train_idx) < min_train:
                    print(f"[P8a][fold-skip] gene={gname} test={len(test_idx)} train={len(train_idx)}")
                    continue
                folds.append((train_idx, test_idx))
            if not folds:
                print("[P8a][folds] No valid LOGO folds — falling back to LOOCV.")
                folds = [(np.array([j for j in range(n) if j != i]), np.array([i])) for i in range(n)]
            return folds

        folds = make_logo_folds_by_gene(gene_ids)

        # --- Optional partial pooling (random-intercept approximation via ridge on group dummies) ---
        # Prefer "gene" if available; otherwise fall back to "target" or "family".
        group_col = None
        for cand in ["gene", "target", "family"]:
            if cand in F_raw_for_relative.columns:
                group_col = cand
                break

        if partial_pooling == "gene_intercepts" and group_col is not None:
            # Build low-variance one-hot intercepts for the grouping column
            PP = add_partial_pooling_X(F_raw_for_relative, group_col)  # returns (n, k) matrix of 0/1 dummies
            # Name the columns and concat to the design matrix
            pp_cols = [f"pp__{group_col}__{g}" for g in pd.Categorical(F_raw_for_relative[group_col]).categories]
            PP_df  = pd.DataFrame(PP, index=X_design.index, columns=pp_cols)
            X_design = pd.concat([X_design, PP_df], axis=1)

            # Persist extended feature order for reproducibility
            feature_order = X_design.columns.tolist()
            with open(MODEL_RUN_DIR/ "feature_order.json", "w") as f:
                json.dump(feature_order, f, indent=2)

        # 8a.3 — Train residualized ridge via your helper
        alpha = float(ridge_alpha)
        summary = fit_ridge_residualized_cv(
            X=X_design.reset_index(drop=True),
            y=y_kd,
            genes=gene_ids,
            families=family_ids,
            folds=folds,
            alpha=alpha
        )

        #print("[P8a] MAE(abs):", summary.mae_abs_mean)
        #print("[P8a] R^2(abs):", summary.r2_abs_mean)
        #print("[P8a] Spearman within-gene (macro):", summary.spearman_within_macro)

        # 8a.4 — Persist outputs compatible with your runners
        p8a_summary_path = MODEL_RUN_DIR / "p8a_summary.json"
        with open(p8a_summary_path, "w") as f:
            json.dump({
                "phase": str(phase),
                "run_id": str(run_id),
                "alpha": alpha,
                "split_mode": "LOGO",
                "mae_abs_mean": summary.mae_abs_mean,
                "r2_abs_mean": summary.r2_abs_mean,
                "spearman_within_macro": summary.spearman_within_macro,
                "per_fold": [fr.__dict__ for fr in summary.per_fold],
            }, f, indent=2)

        # Also write metrics.json so your existing runners can read headline numbers
        metrics = {
            "phase": str(phase),
            "label_transform": "gene_resid",
            "LOOCV_MAE": summary.mae_abs_mean,   # keep the same key names your runner expects
            "LOOCV_R2":  summary.r2_abs_mean,
            "perm_spearman_mean": summary.spearman_within_macro,  # we reuse the field here
            "calibration_slope": None,
            "calibration_intercept": None,
        }
        metrics_path = MODEL_RUN_DIR / "metrics.json"
        with open(metrics_path, "w") as f:
            json.dump(metrics, f, indent=2)
        
        # --- Copy metrics.json to the run dir (residualized) ---
        try:
            (MODEL_RUN_DIR / "metrics.json").write_text(Path(metrics_path).read_text())
        except Exception as e:
            print(f"[P8a-resid][warn] couldn't copy metrics.json: {e}")

        # --- P8a: write features snapshot + run_meta before returning ---
        (FEAT_DIR / "runs" / run_id).mkdir(parents=True, exist_ok=True)
        # snapshot the features this run used; falls back to X if df isn't available
        (F if 'df' not in locals() else df).to_csv(FEAT_DIR / f"runs/{run_id}/features.csv", index=False)

        run_meta = {
            "run_id": run_id,
            "phase": phase, "mode": sim_mode, "relative": relative_eff, "alpha": ridge_alpha,
            "ablation": bool(run_ablation), "use_duplex": use_duplex, "use_access": use_access,
            "synthetic": synthetic_tag,
            "artifacts": {
                "metrics_json": str(metrics_path),
                "p8a_summary_json": str(p8a_summary_path),
                # useful provenance:
                "features_gold_snapshot": str(feat_path) if 'feat_path' in locals() and feat_path is not None else None,
            },
        }
        json.dump(run_meta, open(MODEL_RUN_DIR / "run_meta.json", "w"), indent=2)

        # Early return to skip the standard (non-residualized) training path
        return {
            "run_id": run_id,
            "model_run_dir": str(MODEL_RUN_DIR),
            "fig_run_dir": str(FIG_RUN_DIR),
            "metrics_json": str(metrics_path),
            "coef_csv": None,
            "predictions_csv": None,
            "ablation_csv": None,
            "logo_abs_csv": None,
            "logo_rel_csv": None,
            "calibration_png": None,
        }
    # ---------------- END PHASE 8a conditional ----------------
    
    # ---------------- PHASE 8b — Pairwise ranking within gene (conditional) ----------------
    if label_transform in ("pairwise_gene", "rank_gene", "pairwise"):
        import itertools, json
        from dataclasses import dataclass
        from typing import List, Tuple, Optional
        from sklearn.linear_model import LogisticRegression, LinearRegression
        from sklearn.isotonic import IsotonicRegression
        from scipy.stats import spearmanr
        

        def calibrate_by_group(scores: np.ndarray, y: np.ndarray, group: pd.Series, kind: str = "isotonic"):
            """
            Fit a per-group calibrator on OOF raw scores.
            - isotonic: use 1-D arrays
            - linear:   use 2-D arrays for sklearn LinearRegression
            Returns: (models_dict, global_model, predict_fn)
            """
            kind = (kind or "linear").lower()
            group = pd.Series(group).astype(str).reset_index(drop=True)
            s = np.asarray(scores, dtype=float).ravel()   # 1-D master scores
            t = np.asarray(y, dtype=float).ravel()

            models = {}
            for g, idx in group.groupby(group).groups.items():
                idx = np.array(sorted(list(idx)), dtype=int)
                if len(idx) < 3:
                    continue  # too small → fall back to global
                if kind == "isotonic":
                    m = IsotonicRegression(out_of_bounds="clip")
                    m.fit(s[idx], t[idx])                   # 1-D here
                else:
                    m = LinearRegression()
                    m.fit(s[idx].reshape(-1,1), t[idx])     # 2-D for linear
                models[g] = m

            # global fallback
            if kind == "isotonic":
                global_model = IsotonicRegression(out_of_bounds="clip").fit(s, t)
            else:
                global_model = LinearRegression().fit(s.reshape(-1,1), t)

            def predict(scores_new: np.ndarray, group_new: pd.Series):
                scores_new = np.asarray(scores_new, float).ravel()
                group_new = pd.Series(group_new).astype(str).reset_index(drop=True)
                yhat = np.empty_like(scores_new, dtype=float)
                for g, idx in group_new.groupby(group_new).groups.items():
                    idx = np.array(sorted(list(idx)), dtype=int)
                    model = models.get(g, global_model)
                    # isotonic expects 1-D, linear expects 2-D
                    if isinstance(model, LinearRegression):
                        yhat[idx] = model.predict(scores_new[idx].reshape(-1,1)).ravel()
                    else:
                        yhat[idx] = model.predict(scores_new[idx])
                return yhat

            return models, global_model, predict

        # --- safe identifiers + targets from pre-whitelist frame ---
        def _safe_series(frame: pd.DataFrame, candidates, default):
            for c in candidates:
                if c in frame.columns:
                    return frame[c].astype(str).reset_index(drop=True)
            return pd.Series([default] * len(frame), dtype=str)

        gene_ids   = _safe_series(F_raw_for_relative, ["gene", "target_gene", "target"], default="UNK_GENE")
        family_ids = _safe_series(F_raw_for_relative, ["family", "target_family"], default="UNK_FAM")
        y_kd       = pd.Series(y).astype(float).reset_index(drop=True)

        # --- LOGO-by-gene folds (fallback to LOOCV if necessary) ---
        def make_logo_folds_by_gene(genes: pd.Series, min_train=2, min_test=1):
            folds = []
            n = len(genes)
            groups = genes.groupby(genes).groups
            for gname, idx in groups.items():
                test_idx  = np.array(sorted(list(idx)))
                train_idx = np.array(sorted(list(set(range(n)) - set(test_idx))))
                if len(test_idx) < min_test or len(train_idx) < min_train:
                    continue
                folds.append((train_idx, test_idx, gname))
            if not folds:  # extremely small datasets — degrade gracefully
                folds = [(np.array([j for j in range(n) if j != i]), np.array([i]), genes.iloc[i]) for i in range(n)]
            return folds

        folds = make_logo_folds_by_gene(gene_ids)

        # --- helpers to build pairwise training data within-gene only ---
        def build_pairs(X: np.ndarray, y: np.ndarray, genes: pd.Series, idx: np.ndarray):
            """Within-gene pairwise set from idx. 
            diff = Xi - Xj, label = 1 if yi > yj else 0. Ties skipped."""
            diffs, labels = [], []
            for g in np.unique(genes.iloc[idx]):
                gidx = idx[genes.iloc[idx].values == g]
                if len(gidx) < 2:
                    continue
                for i, j in itertools.combinations(gidx, 2):
                    yi, yj = y[i], y[j]
                    if not (np.isfinite(yi) and np.isfinite(yj)) or yi == yj:
                        continue
                    diffs.append(X[i] - X[j])
                    labels.append(1 if yi > yj else 0)
            if not diffs or len(set(labels)) < 2:
                return None, None
            return np.vstack(diffs), np.array(labels, dtype=int)

        from typing import Optional  # ensure this is imported

        def compute_pair_acc(scores: np.ndarray, y: np.ndarray, genes: pd.Series, test_idx: np.ndarray) -> Optional[float]:
            """Pairwise accuracy across all pairs within the test gene(s)."""
            accs = []
            for g in np.unique(genes.iloc[test_idx]):
                gmask = (genes.iloc[test_idx].values == g)
                ginds = test_idx[gmask]
                if len(ginds) < 2:
                    continue
                correct, total = 0, 0
                for i, j in itertools.combinations(ginds, 2):
                    si, sj = scores[i], scores[j]
                    yi, yj = y[i], y[j]
                    if yi == yj:
                        continue
                    pred  = 1 if si > sj else 0
                    truth = 1 if yi > yj else 0   # flip sign if your KD “lower is better”
                    if pred == truth:
                        correct += 1
                    total += 1
                if total:
                    accs.append(correct / total)
            return float(np.mean(accs)) if accs else None

        # --- map ridge_alpha to logistic C ---
        C = 1.0 if ridge_alpha is None else float(max(1e-3, 1.0 / float(ridge_alpha)))

        X = X_design.reset_index(drop=True).to_numpy(dtype=float)
        y = y_kd.to_numpy(dtype=float)

        # out-of-fold holders
        y_pred = np.full_like(y, fill_value=np.nan, dtype=float)
        s_oo   = np.full_like(y, fill_value=np.nan, dtype=float)  # raw scores before calibration

        per_fold = []
        for train_idx, test_idx, gname in folds:
            # build pairwise training set from TRAIN ONLY
            Xd, yd = build_pairs(X, y, gene_ids, train_idx)

            # --- choose a scorer: logistic weights if we have pairs; otherwise linear baseline ---
            if Xd is not None:
                clf = LogisticRegression(C=C, solver="lbfgs", max_iter=1000)
                clf.fit(Xd, yd)
                w = clf.coef_.ravel()
            else:
                base = LinearRegression()
                base.fit(X[train_idx], y[train_idx])
                w = base.coef_.ravel()  # fall back to linear weights
                # (No early calibration here; we’ll handle it uniformly below.)

            # scores for train/test singletons (uniform path)
            s_train = X[train_idx] @ w
            # If scores are inversely ordered w.r.t KD on TRAIN, flip orientation
            sp_train = spearmanr(s_train, y[train_idx]).correlation
            if sp_train is not None and sp_train < 0:
                w = -w
                s_train = -s_train  # keep s_train consistent after the flip
            s_test  = X[test_idx]  @ w

            # calibrate s -> KD using training fold only
            if calibration == "linear":
                cal = LinearRegression()
                cal.fit(s_train.reshape(-1,1), y[train_idx])
                yhat_test = cal.predict(s_test.reshape(-1,1))
                slope     = float(cal.coef_.ravel()[0])
                intercept = float(cal.intercept_)
            else:  # isotonic
                cal = IsotonicRegression(out_of_bounds="clip")
                cal.fit(s_train, y[train_idx])
                yhat_test = cal.predict(s_test)
                slope, intercept = None, None

            # record out-of-fold preds & scores
            y_pred[test_idx] = yhat_test
            s_oo[test_idx]   = s_test

            # fold metrics
            sp = None
            if len(test_idx) > 1:
                sp = float(spearmanr(y[test_idx], yhat_test).correlation)

            mae = float(np.mean(np.abs(y[test_idx] - yhat_test)))
            pair_acc = compute_pair_acc(scores=s_oo, y=y, genes=gene_ids, test_idx=test_idx)

            per_fold.append({
                "gene": str(gname),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "pair_acc": None if pair_acc is None else float(pair_acc),
                "spearman": None if sp is None else float(sp),
                "mae": float(mae),
                "calibration": calibration,
                "C": float(C),
            })

        # macro metrics across folds with valid values
        spearmans = [d["spearman"] for d in per_fold if d["spearman"] is not None]
        maes      = [d["mae"]      for d in per_fold]
        pairaccs  = [d["pair_acc"] for d in per_fold if d["pair_acc"] is not None]

        logo_macro_spearman = float(np.mean(spearmans)) if spearmans else None
        logo_macro_mae      = float(np.mean(maes))      if maes      else None
        pair_acc_mean       = float(np.mean(pairaccs))  if pairaccs  else None

        # --- Optional: override fold-wise y_pred with grouped calibration on pooled out-of-fold scores ---
        scope = calibration_scope
        def _safe_grouped_calibrate(scores, y_true, group_series, kind):
            s = np.asarray(scores, float).ravel()
            t = np.asarray(y_true, float).ravel()
            g = pd.Series(group_series).astype(str).reset_index(drop=True)
            # mask valid
            mask = np.isfinite(s) & np.isfinite(t) & g.notna()
            if mask.sum() < 3:
                # not enough points to fit a grouped calibrator—skip override
                return None
            # fit on the masked subset
            models, global_model, _predict = calibrate_by_group(s[mask], t[mask], g[mask], kind=kind)
            # predict only on valid indices; keep existing y_pred elsewhere
            def _predict_full(s_all, g_all):
                s_all = np.asarray(s_all, float).ravel()
                g_all = pd.Series(g_all).astype(str).reset_index(drop=True)
                yhat = np.array(y_pred, float)  # start from per-fold
                if mask.any():
                    # build per-group indices on the full array
                    idx_valid = np.where(np.isfinite(s_all) & np.isfinite(t))[0]
                    if idx_valid.size:
                        # grouped predict on valid rows only
                        yhat[idx_valid] = _predict(s_all[idx_valid], g_all[idx_valid]).ravel()
                return yhat
            return _predict_full

        if scope == "by_family" and "family" in F_raw_for_relative.columns:
            pred_fn = _safe_grouped_calibrate(s_oo, y, F_raw_for_relative["family"], kind=calibration)
            if pred_fn is not None:
                y_pred = pred_fn(s_oo, F_raw_for_relative["family"])
        elif scope == "by_gene" and "gene" in F_raw_for_relative.columns:
            pred_fn = _safe_grouped_calibrate(s_oo, y, F_raw_for_relative["gene"], kind=calibration)
            if pred_fn is not None:
                y_pred = pred_fn(s_oo, F_raw_for_relative["gene"])
        # else: keep the per-fold calibrated y_pred as-is
        
        # ==== NEW: global metrics on final y_pred (what predictions.csv contains) ====
        def _safe_r2(y_true, y_hat):
            num = float(np.sum((y_true - y_hat)**2))
            den = float(np.sum((y_true - np.mean(y_true))**2))
            return float(1.0 - num/den) if den > 0 else None

        from scipy.stats import spearmanr
        mae_all = float(np.mean(np.abs(y - y_pred)))
        r2_all  = _safe_r2(y, y_pred)
        sp_all  = float(spearmanr(y, y_pred).correlation) if len(y) > 1 else None
        # ==============================================
        
        # persist predictions & summaries
        pred_df = pd.DataFrame({
            "index": np.arange(len(y)),
            "gene": gene_ids,
            "family": family_ids,
            "KD_true": y_kd,
            "score_raw": s_oo,
            "KD_pred_cal": y_pred,
        })
        pred_csv = MODEL_RUN_DIR / "predictions.csv"
        pred_df.to_csv(pred_csv, index=False)

        # headline metrics for runners
        metrics = {
            "phase": str(phase),
            "label_transform": "pairwise_gene",
            "calibration": calibration,
            "C": C,
            "pair_acc_mean": pair_acc_mean,
            "logo_macro_spearman": logo_macro_spearman,
            "logo_macro_mae": logo_macro_mae,
            # keep common keys so upstream dashboards don't break
            "LOOCV_MAE": mae_all,
            "LOOCV_R2":  r2_all,
            "perm_spearman_mean": sp_all,
            "calibration_slope": None,  # populated only for linear
            "calibration_intercept": None,
            "calibration_scope": calibration_scope,
        }
        if calibration == "linear":
            # last fold's slope/intercept aren't meaningful across folds; leave None or compute by refit on full data if needed
            pass

        metrics_path = MODEL_RUN_DIR / "metrics.json"
        with open(metrics_path, "w") as f:
            json.dump(metrics, f, indent=2)

        # small LOGO summary for parity with P8a
        with open(MODEL_RUN_DIR / "logo_abs_summary.json", "w") as f:
            json.dump({
                "logo_macro_spearman": logo_macro_spearman,
                "logo_macro_mae": logo_macro_mae,
                "pair_acc_mean": pair_acc_mean
            }, f, indent=2)

        p8b_summary_path = MODEL_RUN_DIR / "p8b_summary.json"
        with open(p8b_summary_path, "w") as f:
            json.dump({
                "phase": str(phase),
                "run_id": str(run_id),
                "calibration": calibration,
                "calibration_scope": calibration_scope, 
                "C": C,
                "per_fold": per_fold,
            }, f, indent=2)
        
        # --- P8b: write features snapshot + run_meta before returning ---
        (FEAT_DIR / "runs" / run_id).mkdir(parents=True, exist_ok=True)
        (F if 'df' not in locals() else df).to_csv(FEAT_DIR / f"runs/{run_id}/features.csv", index=False)

        # features snapshot + run_meta
        (FEAT_DIR / "runs" / run_id).mkdir(parents=True, exist_ok=True)
        (F if 'df' not in locals() else df).to_csv(FEAT_DIR / f"runs/{run_id}/features.csv", index=False)
        json.dump({
            "run_id": run_id, "phase": phase, "mode": sim_mode, "relative": relative_eff,
            "alpha": ridge_alpha, "ablation": bool(run_ablation),
            "use_duplex": use_duplex, "use_access": use_access, "synthetic": synthetic_tag,
            "calibration": calibration,
            "artifacts": {"metrics_json": str(metrics_path), "predictions_csv": str(pred_csv),
                        "p8b_summary_json": str(p8b_summary_path),
                        "features_gold_snapshot": str(feat_path) if 'feat_path' in locals() and feat_path is not None else None}
        }, open(MODEL_RUN_DIR / "run_meta.json", "w"), indent=2)

        return {
            "run_id": run_id,
            "model_run_dir": str(MODEL_RUN_DIR),
            "fig_run_dir": str(FIG_RUN_DIR),
            "metrics_json": str(metrics_path),
            "coef_csv": None,
            "predictions_csv": str(pred_csv),
            "ablation_csv": None,
            "logo_abs_csv": None,
            "logo_rel_csv": None,
            "calibration_png": None,
        }
    # ---------------- END PHASE 8b ----------------
    
    # ---------------- STEP 5 — Synthetic data (if sim_mode == "synthetic") ----------------
    if sim_mode == "synthetic":
        X_train, y_train, w_train = make_synthetic_training(
            F=F,
            X_design=X_design,
            y=y,
            model_dir=MODEL_DIR,
            feature_order=feature_order,
            relative=relative_eff,
            ridge_alpha=ridge_alpha,
            synth_n_per_real=synth_n_per_real,
            synth_weight=synth_weight,
            noise_sigma0=noise_sigma0,
            noise_c1=noise_c1,
            noise_c2=noise_c2,
            seed=42,
        )
        # Belt & suspenders: force schema to exactly the feature_order from Step-4
        X_train = X_train.reindex(columns=feature_order, fill_value=0.0)
    else:
        X_train, y_train, w_train = X_design, y, np.ones(len(y), dtype=float)
        
    #print("[debug] calling LOOCV with columns:", list(X_train.columns))
    #print("[debug] shapes  X_train/y_train/w_train:", X_train.shape, len(y_train), len(w_train))

    # ---------------- STEP 6 — Fit & LOOCV over REAL rows only ----------------
    _ = fit_ridge_logit(X_train, y_train, w_train, ridge_alpha=ridge_alpha)
    
    #print("[debug] calling LOOCV with columns:", list(X_train.columns))

    preds, metrics_path, coef_path = loocv_metrics_real_only(
        model_dir=MODEL_RUN_DIR,
        sim_mode=sim_mode,
        X_train=X_train,   # <-- filtered matrix only
        y_train=y_train,
        w_train=w_train,    # or None
        y_real_len=len(y),          # the first len(y) rows are the real samples
        ridge_alpha=ridge_alpha,
        verbose=verbose,
    )

    ## Per-ASO predictions
    # Pull IDs from the raw frame (has aso_id/target), not from X_design
    aso_col    = "aso_id" if "aso_id" in F_raw_for_relative.columns else None
    target_col = "target" if "target" in F_raw_for_relative.columns else None

    pred_df = pd.DataFrame({
        # Use raw IDs if available; otherwise fall back to sequence index
        "aso_id":  (F_raw_for_relative[aso_col].reset_index(drop=True)
                    if aso_col else pd.Series(range(1, len(y)+1))),
        "target":  (F_raw_for_relative[target_col].reset_index(drop=True)
                    if target_col else pd.Series(["NA"] * len(y))),
        "KD_true": y,
        "KD_pred_LOOCV": preds,
        "mode": sim_mode,
        "relative": relative_eff,   # use effective flag
        "alpha": ridge_alpha,
        "neg_dG_bind_source": ("thermo" if sim_mode in {"enhanced","synthetic"} else "proxy"),
        "use_duplex": use_duplex,
        "use_access": use_access,
        "synthetic": synthetic_tag,
        "synth_n_per_real": (synth_n_per_real if sim_mode == "synthetic" else None),
        "synth_weight": (synth_weight if sim_mode == "synthetic" else None),
    })

    pred_path = MODEL_RUN_DIR / "predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    # Augment metrics (RMSE, calibration, permutation baseline)
    with open(metrics_path, "r") as f:
        mets = json.load(f)
    mets["RMSE"] = rmse(y, preds)
    c0, c1 = calibration_stats(y, preds)
    mets["calibration_intercept"] = c0
    mets["calibration_slope"]     = c1
    mets.update(permutation_baseline_spearman(y, n=300, seed=0))
    with open(metrics_path, "w") as f:
        json.dump(mets, f, indent=2)
        
    # --- Ensure metrics.json exists in the RUN DIR (identity) ---
    import json, numpy as np
    from scipy.stats import spearmanr

    metrics_dst = MODEL_RUN_DIR / "metrics.json"
    data = {}
    # try to copy the file we just updated
    try:
        data = json.loads(Path(metrics_path).read_text())
    except Exception:
        data = {}

    # if minimal fields are missing, compute from y & preds
    if "LOOCV_MAE" not in data or "logo_macro_mae" not in data or "logo_macro_spearman" not in data:
        mae = float(np.mean(np.abs(y - preds)))
        rho = float(spearmanr(y, preds).correlation)
        data.setdefault("LOOCV_MAE", mae)
        data.setdefault("logo_macro_mae", data.get("LOOCV_MAE", mae))
        data.setdefault("logo_macro_spearman", rho)
        data.setdefault("perm_spearman_mean", rho)

    data.setdefault("phase", str(phase))           # "P8a"
    data.setdefault("label_transform", "identity")

    metrics_dst.write_text(json.dumps(data, indent=2))

    # ---------------- STEP 7a — Relative-KD evaluation (optional) ----------------
    rel_loocv_df = rel_logo_df = None
    rel_logo_csv_path = None
    if relative_eff:
        rel_loocv_df, rel_logo_df, rel_logo_csv_path = relative_loocv_logo(
            F=F_raw_for_relative,
            model_dir=MODEL_RUN_DIR,
            ridge_alpha=ridge_alpha,
            sim_mode=sim_mode,
            verbose=verbose,
        )

    # ---------------- STEP 7b — Ablation and absolute LOGO Spearman ----------------
    abl_csv_path, logo_csv_path = run_ablation_and_logo(
        F=F_raw_for_relative,
        X_design=X_design,
        core_feats=core_feats,
        ridge_alpha=ridge_alpha,
        run_ablation=run_ablation,
        run_logo_flag=run_logo,
        model_dir=MODEL_RUN_DIR,
        fig_dir=FIG_DIR,
        sim_mode=sim_mode,
        relative=relative_eff,
        verbose=verbose,
    )
    
    # Summaries
    logo_abs_summary_path = None
    if logo_csv_path:
        logo_df = pd.read_csv(logo_csv_path)
        js = summarize_logo_csv(logo_df)
        logo_abs_summary_path = MODEL_RUN_DIR / "logo_abs_summary.json"
        json.dump(js, open(logo_abs_summary_path, "w"), indent=2)

    logo_rel_summary_path = None
    if rel_logo_csv_path:
        rel_logo_df = pd.read_csv(rel_logo_csv_path)
        js = summarize_logo_csv(rel_logo_df)
        logo_rel_summary_path = MODEL_RUN_DIR / "logo_rel_summary.json"
        json.dump(js, open(logo_rel_summary_path, "w"), indent=2)

    # ---------------- STEP 7c — Calibration plot ----------------
    calib_path = save_calibration_plot(
        y_true=y,
        y_pred=preds,
        fig_dir=FIG_DIR,
        sim_mode=sim_mode,
        relative=relative_eff,
    )

    # Copy features used by this run into a per-run snapshot (provenance)
    (FEAT_DIR / "runs" / run_id).mkdir(parents=True, exist_ok=True)
    (df if 'df' in locals() else F).to_csv(FEAT_DIR / f"runs/{run_id}/features.csv", index=False)

    # ---------------- Run metadata ----------------
    run_meta = {
        "run_id": run_id,
        "phase": phase, "mode": sim_mode, "relative": relative_eff, "alpha": ridge_alpha, "ablation": bool(run_ablation),
        "neg_dG_bind_source": ("thermo" if sim_mode in {"enhanced","synthetic"} else "proxy"),
        "use_duplex": use_duplex, "use_access": use_access,
        "synthetic": synthetic_tag,
        "synth_n_per_real": (synth_n_per_real if sim_mode=="synthetic" else None),
        "synth_weight": (synth_weight if sim_mode=="synthetic" else None),
        "artifacts": {
            "metrics_json": str(metrics_path),
            "coef_csv": str(coef_path),
            "predictions_csv": str(pred_path),
            "ablation_csv": abl_csv_path and str(abl_csv_path),
            "logo_abs_csv": logo_csv_path and str(logo_csv_path),
            "logo_rel_csv": rel_logo_csv_path and str(rel_logo_csv_path),
            "logo_abs_summary_json": logo_abs_summary_path and str(logo_abs_summary_path),
            "logo_rel_summary_json": logo_rel_summary_path and str(logo_rel_summary_path),
            "calibration_png": str(calib_path),
            # NEW keys for traceability
            "features_step2_snapshot": str(snap2) if snap2 is not None else None,
            "features_gold_snapshot":  str(gold_path) if gold_path is not None else None,
            "feat_path_used":          str(feat_path),
                }
    }
    json.dump(run_meta, open(MODEL_RUN_DIR / "run_meta.json", "w"), indent=2)

    from datetime import datetime
    import json, shutil

    # ---------- SNAPSHOT (after successful run) ----------
    # robust run_id fallback
    run_id_str = str(run_id) if "run_id" in locals() else MODEL_RUN_DIR.name
    SNAP_DIR = FEAT_DIR / "snapshots" / "runs" / run_id_str
    SNAP_DIR.mkdir(parents=True, exist_ok=True)

    # 1) Feature snapshot (use the full features frame returned by design builder)
    try:
        F.to_csv(SNAP_DIR / "features.csv", index=False)
    except Exception as e:
        print("[warn] could not snapshot features.csv:", e)

    # 2) Run meta (so you can reproduce)
    meta = {
        "phase": str(phase),
        "feature_set": str(feature_set or basic_feature_set),
        "calibration": str(calibration),
        "calibration_scope": str(calibration_scope),
        "ridge_alpha": float(ridge_alpha) if "ridge_alpha" in locals() else None,
        "partial_pooling": partial_pooling,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model_run_dir": str(MODEL_RUN_DIR),
    }
    json.dump(meta, open(SNAP_DIR / "run_meta.json", "w"), indent=2)

    # 3) Copy metrics/coefs if they exist
    try:
        metrics_path = Path(MODEL_RUN_DIR) / "metrics.json"
        if metrics_path.exists():
            shutil.copy2(metrics_path, SNAP_DIR / "metrics.json")
    except Exception as e:
        print("[warn] could not copy metrics.json:", e)

    try:
        coef_path = Path(MODEL_RUN_DIR) / "coef.csv"
        if coef_path.exists():
            shutil.copy2(coef_path, SNAP_DIR / "coef.csv")
    except Exception as e:
        print("[warn] could not copy coef.csv:", e)
# ---------- end snapshot ----------
    return {
        "run_id": run_id,
        "model_run_dir": str(MODEL_RUN_DIR),
        "fig_run_dir": str(FIG_RUN_DIR),
        "features_csv": str(feat_path),
        "metrics_json": str(metrics_path),
        "coef_csv": str(coef_path),
        "predictions_csv": str(pred_path),
        "ablation_csv": abl_csv_path and str(abl_csv_path),
        "logo_abs_csv": logo_csv_path and str(logo_csv_path),
        "logo_rel_csv": rel_logo_csv_path and str(rel_logo_csv_path),
        "calibration_png": str(calib_path),
        "n_real": int(len(y)),
        "n_train_total": int(len(X_train)),
    }


In [7]:
# === P10b Helper functions  ===
from pathlib import Path
import pandas as pd
import numpy as np
import json, time, shutil

def _read_any(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Snapshot not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    elif path.suffix.lower() in (".parquet", ".pq"):
        # optional: if you later install pyarrow/fastparquet this will work
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported snapshot format: {path.suffix}")

# Re-define _merge_snapshot to accept CSV/parquet
def _merge_snapshot(df: pd.DataFrame, snap_path: Path) -> pd.DataFrame:
    if not snap_path or not Path(snap_path).exists():
        return df
    snap = _read_any(snap_path)
    if "aso_id_norm" in df.columns and "aso_id_norm" in snap.columns:
        key = "aso_id_norm"
    elif all(c in df.columns for c in ("aso_seq","target_seq")) and \
         all(c in snap.columns for c in ("aso_seq","target_seq")):
        key = ["aso_seq","target_seq"]
    else:
        return df
    return df.merge(snap.drop_duplicates(subset=key), on=key, how="left")
def _add_thermo_fields(df):
    out = df.copy()
    if "dG37_kcalmol" not in out.columns:
        raise ValueError("Missing dG37_kcalmol. Merge ViennaRNA duplex ΔG°37 first.")
    if "dG_open_mean" not in out.columns:
        out["dG_open_mean"] = 0.0
    out["neg_dG_bind"] = -out["dG37_kcalmol"]
    out["dG_eff"]      = out["dG37_kcalmol"] - out["dG_open_mean"]
    out["neg_dG_eff"]  = -out["dG_eff"]
    for col in ("rnaseh_heur","open_state_persist"):
        if col not in out.columns:
            out[col] = 0.0
    return out

def _seed_features(df, ks=(5,6)):
    if not all(c in df.columns for c in ("aso_seq","target_seq")):
        for k in ks:
            df[f"seed_k{k}_hits"] = 0
            df[f"seed_k{k}_frac"] = 0.0
        return df
    out = df.copy()
    def kmers(s,k): return {s[i:i+k] for i in range(0, max(0,len(s)-k+1))}
    for k in ks:
        hits, fracs = [], []
        for aso, tgt in zip(out["aso_seq"].astype(str), out["target_seq"].astype(str)):
            if len(aso)<k or len(tgt)<k:
                hits.append(0); fracs.append(0.0); continue
            A, T = kmers(aso.upper(),k), kmers(tgt.upper(),k)
            h = len(A & T); hits.append(h); fracs.append(h/max(1,len(A)))
        out[f"seed_k{k}_hits"] = hits
        out[f"seed_k{k}_frac"] = fracs
    return out

def _add_thermo_fields__lenient(df: pd.DataFrame):
    out = df.copy()
    if "dG37_kcalmol" in out.columns:
        if "dG_open_mean" not in out.columns:
            out["dG_open_mean"] = 0.0
        out["neg_dG_bind"] = -out["dG37_kcalmol"]
        out["dG_eff"]      = out["dG37_kcalmol"] - out["dG_open_mean"]
        out["neg_dG_eff"]  = -out["dG_eff"]
    for col in ("rnaseh_heur","open_state_persist"):
        if col not in out.columns:
            out[col] = 0.0
    return out

def build_P10b_splits(
    train_csv=None,
    val_csv=None,
    phase4_snapshot=None,
    p10b_snapshot=None
):
    if train_csv is None:
        train_csv = FEAT_DIR / "external_fast" / "train_basic.csv"

    if val_csv is None:
        val_csv = FEAT_DIR / "external_fast" / "val_basic.csv"
        
    if phase4_snapshot is None:
            phase4_snapshot = FEAT_DIR / "snapshots" / "aso_features_step2__phase1_fixed_with_thermo__p9.csv"
        
    if p10b_snapshot is None:
        p10b_snapshot = FEAT_DIR / "snapshots" / "P10b_fullbio_snapshot.csv"
        
    train = pd.read_csv(train_csv)
    val   = pd.read_csv(val_csv)

    snap_in = Path(phase4_snapshot) if Path(phase4_snapshot).exists() else None
    train_f = _merge_snapshot(train, snap_in)
    val_f   = _merge_snapshot(val,   snap_in)

    # cheap seeds
    train_f = _seed_features(train_f)
    val_f   = _seed_features(val_f)

    # thermo derived fields if available
    have_thermo = ("dG37_kcalmol" in train_f.columns) and ("dG37_kcalmol" in val_f.columns)
    train_f = _add_thermo_fields__lenient(train_f)
    val_f   = _add_thermo_fields__lenient(val_f)

    # write CSV snapshot (no parquet)
    Path(p10b_snapshot).parent.mkdir(parents=True, exist_ok=True)
    pd.concat([train_f.assign(_split="train"), val_f.assign(_split="val")]) \
      .to_csv(p10b_snapshot, index=False)

    # write splits for training
    train_out = FEAT_DIR / "external_fast" / "train_thermo.csv"
    val_out   = FEAT_DIR / "external_fast" / "val_thermo.csv"
    train_f.to_csv(train_out, index=False)
    val_f.to_csv(val_out,   index=False)

    return train_out, val_out, str(p10b_snapshot), have_thermo
def _rename_latest_run(prefix: str):
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    cands = sorted([d for d in RUNS_DIR.glob("*") if d.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    latest = next((d for d in cands if (d/"predictions.csv").exists()), None)
    if latest is None:
        return None
    new = RUNS_DIR / prefix
    if new.exists():
        shutil.rmtree(new)
    latest.rename(new)
    return new

# Rebind run_pipeline_P10b to accept CSV snapshot and avoid parquet writes
def run_pipeline_P10b(
    alphas=(0.3, 1.0, 3.0),
    mech_opts=(0, 1),
    use_phase4_snapshot=True,
    phase4_snapshot="features/snapshots/aso_features_phase10b_with_thermo.csv",
    feat_path=None,
):
    tr_csv, va_csv, snap, have_thermo = build_P10b_splits(
        phase4_snapshot=phase4_snapshot if use_phase4_snapshot else "___none___"
    )
    print(f"[P10b] splits → {tr_csv}, {va_csv}")
    print(f"[P10b] snapshot (CSV) → {snap}")
    print(f"[P10b] have_thermo_in_splits={have_thermo}")

    rows = []
    for a in alphas:
        for m in mech_opts:
            tag = f"P10b__ordering_THERMO__a{a}__mech{m}"
            print(f"\n>>> Running {tag}")
            run_pipeline(
                sim_mode="enhanced",
                relative=False,
                ridge_alpha=float(a),
                run_ablation=False,
                run_logo=False,
                full_bio=True,
                thermo_cache=True,
                force_recompute_thermo=False,  # compute inside if snapshot lacked dG37
                ROOT=ROOT,
                calibration="isotonic",           
                calibration_scope="by_family", 
                feat_path_override=str(feat_path) if feat_path else str(phase4_snapshot),
                # If run_pipeline supports explicit train/val kwargs, pass them here:
                # train_csv=tr_csv, val_csv=va_csv,
            )
            renamed = _rename_latest_run(tag)
            mets = {}
            if renamed and (renamed/"metrics.json").exists():
                try:
                    mets = json.loads((renamed/"metrics.json").read_text())
                except Exception as e:
                    mets = {"ERROR": str(e)}
            rows.append({"alpha": a, "mech": m, "run_dir": str(renamed) if renamed else "NA", **mets})
            print(f"[done] → {renamed}")

    import pandas as pd, time
    df = pd.DataFrame(rows)
    keep = [c for c in ("alpha","mech","MAE","R2","Spearman","bias","run_dir") if c in df.columns]
    df = df[keep] if keep else df
    ts = time.strftime("%Y%m%d_%H%M%S")
    out = RUNS_DIR / f"P10b__summary__{ts}.json"
    out.write_text(json.dumps(df.to_dict(orient="records"), indent=2))
    print(f"[P10b] summary → {out}")
    return df


In [8]:
# --- Thermo preflight: catch errors early (seconds, not minutes) ---
from pathlib import Path
import pandas as pd
import thermo

feat_p = (ROOT / "features" / "aso_features.csv")
assert feat_p.exists(), f"Missing features file: {feat_p}. Run until [features] wrote ... first."

feats = pd.read_csv(feat_p)

aso_col  = "sequence" if "sequence" in feats.columns else ("Sequence" if "Sequence" in feats.columns else None)
targ_col = "target_window" if "target_window" in feats.columns else None
print("[preflight] ASO_COL:", aso_col, "TARG_COL:", targ_col)
if aso_col is None or targ_col is None:
    raise SystemExit(f"[preflight] Required columns not found. Available: {list(feats.columns)}")

# enforce overrides the same way thermo.py expects them
thermo.THERMO_ASO_COL_OVERRIDE    = aso_col
thermo.THERMO_TARGET_COL_OVERRIDE = targ_col
print("[preflight] thermo overrides:", thermo.THERMO_ASO_COL_OVERRIDE, thermo.THERMO_TARGET_COL_OVERRIDE)

# sanity: at least one row has both sequences
m = feats[aso_col].notna() & feats[targ_col].notna() & (feats[aso_col].str.len() > 0) & (feats[targ_col].str.len() > 0)
print("[preflight] rows with ASO+TARGET present:", int(m.sum()), "/", len(feats))
if m.sum() == 0:
    raise SystemExit("[preflight] No rows with both ASO and TARGET sequences present.")

# Optional: show a couple of example lengths
print(feats.loc[m, [aso_col, targ_col]].head(30).assign(
    aso_len=lambda d: d[aso_col].str.len(),
    targ_len=lambda d: d[targ_col].str.len()
))


[preflight] ASO_COL: sequence TARG_COL: target_window
[preflight] thermo overrides: sequence target_window
[preflight] rows with ASO+TARGET present: 8 / 8
               sequence         target_window  aso_len  targ_len
0  TCTCTAACCCATCAGCACAA  UCUCUAACCCAUCAGCACAA       20        20
1  CTCTAACCCATCAGCACAAT  CUCUAACCCAUCAGCACAAU       20        20
2  CAGAACCACAACGTGCAAGG  CAGAACCACAACGUGCAAGG       20        20
3  AGAACCACAACGTGCAAGGG  GAACCACAACGUGCAAGGGU       20        20
4  GAACCACAACGTGCAAGGGT  GAACCACAACGUGCAAGGGU       20        20
5  AAGCCACTCAAGTCCTGCCC  GGGCAGGACUUGAGUGGCUU       20        20
6  ATTCCGAAAGAATGGTCACA  UGUGACCAUUCUUUCGGAAU       20        20
7  TGAGTGTGCCCATGAGTGTC  GAGUGUGCCCAUGAGUGUCU       20        20


In [9]:
# === Define missing constants used inside run_pipeline ===
from pathlib import Path

# Used by run_pipeline to name the “upgraded/normalized” snapshot outputs
NORMALIZED_SNAPSHOT_SUFFIX = "__normalized.csv"
FIG_RUN_DIR = FIG_DIR 

print("Set NORMALIZED_SNAPSHOT_SUFFIX =", NORMALIZED_SNAPSHOT_SUFFIX)
print("FIG_RUN_DIR ->", FIG_RUN_DIR)


Set NORMALIZED_SNAPSHOT_SUFFIX = __normalized.csv
FIG_RUN_DIR -> /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/outputs/figures


In [10]:
from pathlib import Path
import pandas as pd

FEAT_CSV = FEAT_DIR / "aso_features.csv"
if not FEAT_CSV.exists():
    raise FileNotFoundError(
        f"Required feature table not found: {FEAT_CSV}"
    )
_preview = pd.read_csv(FEAT_CSV, nrows=3)

print("[features] Using:", FEAT_CSV)
print("Columns:", list(_preview.columns))

[features] Using: /mnt/c/Users/HP/Desktop/folder to test igem model sharing for Frontiers reviewers/igemModel copy/features/aso_features.csv
Columns: ['target', 'aso_id', 'group', 'rel_expr', 'KD', 'sequence', 'Sequence', 'target_window', 'target_start_idx', 'mapped_transcript', 'mapping_mismatches', 'p_unpaired_site', 'logit_punp', 'duplex_dG', 'neg_duplex_dG', 'dG_open_mean', 'dG_open_min', 'dG_eff', 'neg_dG_eff', 'aso_gc_frac', 'target_gc_pm10', 'target_gc_pm30', 'kmer_pca1', 'kmer_pca2', 'kmer_pca3', 'kmer_pca4', 'kmer_pca5', 'kmer_pca6', 'kmer_pca7', 'neg_dG_bind', 'OT_weighted', 'GC_pen', 'PC']


In [11]:
# --- Pre-sweep: bind to cached snapshots, assert columns, never rebuild ---
from pathlib import Path
import pandas as pd

SNAP_DIR = FEAT_DIR / "snapshots"

THERMO_20 = SNAP_DIR / "thermo_base_20mer.csv"          # baseline 20-mer thermo
THERMO_23 = SNAP_DIR / "thermo_plus_kmer23.csv"         # 2/3-mer PCs
THERMO_56 = SNAP_DIR / "thermo_plus_kmer56.csv"         # 5/6-mer PCs (fixed)
PHASE10B_THERMO = SNAP_DIR / "aso_features_phase10b_with_thermo.csv"  # proxy/orig (fixed)

SNAP_PATHS = {
    "proxy_only": PHASE10B_THERMO,
    "proxy_duplex": PHASE10B_THERMO,
    "proxy_duplex_access": PHASE10B_THERMO,
    "thermo_base_20mer": THERMO_20,
    "thermo_plus_kmer23": THERMO_23,
    "thermo_plus_kmer56": THERMO_56,
}

# Strict "must-have" columns
REQUIRED_COLS = {
    "proxy_only": {"neg_dG_bind"},
    "proxy_duplex": {"neg_dG_bind", "neg_duplex_dG"},
    # kmer23 uses unified PCs kmer_pca1..N in your logs
    "thermo_base_20mer": {"neg_dG_bind", "neg_duplex_dG"},  # access optional via ANY_OF
    "thermo_plus_kmer23": {"neg_dG_bind", "neg_duplex_dG", "kmer_pca1"},
    # kmer56 uses kmer5_PC*, kmer6_PC* in your logs
    "thermo_plus_kmer56": {"neg_dG_bind", "neg_duplex_dG", "kmer5_PC1", "kmer6_PC1"},
    # proxy_duplex_access: access feature required but allow either logit OR p_unpaired
    "proxy_duplex_access": {"neg_dG_bind", "neg_duplex_dG"},
}

# Flexible "any-of" requirements (pass if ANY present)
ANY_OF = {
    "proxy_duplex_access": [{"logit_punp", "p_unpaired_site"}],
    "thermo_base_20mer":   [{"logit_punp", "p_unpaired_site"}],  # optional but nice to assert if present
}

def _assert_has(path: Path, must_have: set, any_of_reqs=None):
    cols = set(pd.read_csv(path, nrows=1).columns)
    missing = sorted([c for c in must_have if c not in cols])
    any_missing = []
    if any_of_reqs:
        for group in any_of_reqs:
            if cols.isdisjoint(group):
                any_missing.append(sorted(group))
    if missing or any_missing:
        msg = [f"{path.name} failed audit:"]
        if missing:
            msg.append(f"  missing required: {missing}")
        if any_missing:
            msg.append(f"  none of present from groups: {any_missing}")
        raise RuntimeError("\n".join(msg))

# Audit once so the sweep can safely use FAST PATH loads
for fs_name, csv_path in SNAP_PATHS.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"Snapshot not found for {fs_name}: {csv_path}")
    _assert_has(csv_path, REQUIRED_COLS.get(fs_name, set()), ANY_OF.get(fs_name))

print("✓ Snapshot cache audit OK — sweep will use FAST PATH loads only.")

USE_SNAPSHOT_CACHE = True


✓ Snapshot cache audit OK — sweep will use FAST PATH loads only.


In [13]:
# ============================================================
# Export Figure 4 ablation summary
# ============================================================
import pandas as pd

fig4_ablation = pd.DataFrame({
    "short": [
        "Sequence-only",
        "+Thermodynamics",
        "+Thermodynamics +Accessibility",
    ],
    "long": [
        "Sequence only (k-mers, GC, wing/gap GC)",
        "RNA:DNA NN thermodynamics + binding-site mapping",
        "RNAplfold ED + ΔG_eff; ATXN2-only (best transfer)",
    ],
    "MAE_wetlab": [
        19.90,
        11.50,
        10.00,
    ],
})

fig4_csv = RESULTS_DIR_BASE / "fig4_ablation_values.csv"
fig4_ablation.to_csv(
    fig4_csv,
    index=False
)

